# BeamLabStudio — Quickstart

This notebook demonstrates the core functionality of BeamLabStudio's Python API.

**Prerequisites:**
- `pip install beamlab` (or `cmake --build build --target pybeamlab`)
- NumPy, Matplotlib, Pandas

**Contents:**
1. Loading material and particle registries
2. Bragg curve simulation (proton in water)
3. Profile-based analysis configuration
4. Batch processing multiple CSV files
5. NIST validation

In [ ]:
# ── Imports ──────────────────────────────────────────────────────
import beamlab
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import time

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Material and Particle Registries

BeamLabStudio ships with 55+ ICRU/NIST materials and 23 PDG 2022 particle species.
Registries are populated lazily via `load_builtin()`.

In [ ]:
# ── Material Registry ────────────────────────────────────────────
mats = beamlab.MaterialRegistry()
mats.load_builtin()

print(f"Loaded {len(mats)} materials:")
print(f"  Names: {mats.names()[:5]}...")

# Access by ID (O(1)).
water = mats['water_icru']
print(f"  Water:  ρ={water.density_g_cm3:.3f} g/cm³  "
      f"I={water.meanExcitationEnergy_eV:.1f} eV  "
      f"X0={water.radiationLength_cm:.2f} cm")

lead = mats['lead']
print(f"  Lead:   ρ={lead.density_g_cm3:.2f} g/cm³  "
      f"I={lead.meanExcitationEnergy_eV:.1f} eV  "
      f"X0={lead.radiationLength_cm:.4f} cm")

In [ ]:
# ── Particle Registry ────────────────────────────────────────────
parts = beamlab.ParticleRegistry()
parts.load_builtin()

print(f"Loaded {len(parts)} particles:")
print(f"  Names: {parts.names()[:5]}...")

# Access by PDG code.
p = parts.get_by_pdg(2212)
print(f"  Proton:   mass={p.mass_MeV:.4f} MeV  charge={p.charge_e:+g}  "
      f"WR={p.WR}")

e = parts.get_by_pdg(11)
print(f"  Electron: mass={e.mass_MeV:.6f} MeV  charge={e.charge_e:+g}")

alpha = parts.get_by_pdg(1000020040)
print(f"  Alpha:    mass={alpha.mass_MeV:.2f} MeV  charge={alpha.charge_e:+g}  "
      f"WR={alpha.WR}")

## 2. Bragg Curve Simulation

Compute the depth-dose profile (Bragg curve) for a 150 MeV proton beam
in water. The `SimulationEngine` integrates `computeStep()` iteratively
until the particle stops (kinetic energy < 0.1 MeV).

In [ ]:
# ── Simulation Engine ────────────────────────────────────────────
engine = beamlab.SimulationEngine(mats, parts)

# Single step calculation.
step = engine.compute_step(150.0, stepLength_cm=0.01,
                           material_name='water_icru',
                           particle_name='proton')
print(f"dE/dx = {step.dEdx_MeV_cm:.3f} MeV/cm")
print(f"MCS angle = {step.mcsAngle_rad:.5f} rad")
print(f"Straggling sigma = {step.stragglingSigma_MeV:.5f} MeV")

In [ ]:
# ── Full Bragg curve ─────────────────────────────────────────────
t0 = time.time()
curve = engine.compute_bragg_curve(150.0, 'water_icru', 'proton')
t1 = time.time()

print(f"Computed {len(curve.depth_cm)} steps in {t1-t0:.3f}s")
print(f"Peak at z = {curve.peakDepth_cm:.3f} cm")
print(f"Peak dE/dx = {curve.peakDEdx_MeV_cm:.2f} MeV/cm")
print(f"Range = {curve.depth_cm[-1]:.3f} cm")

In [ ]:
# ── Plot Bragg curves for multiple energies ──────────────────────
plt.figure(figsize=(10, 5))

for E_MeV in [50, 100, 150, 200, 250]:
    c = engine.compute_bragg_curve(E_MeV, 'water_icru', 'proton')
    plt.plot(c.depth_cm, c.dEdx_MeV_cm,
             label=f'{E_MeV} MeV (peak z={c.peakDepth_cm:.1f} cm)')

plt.xlabel('Depth [cm]')
plt.ylabel('dE/dx [MeV/cm]')
plt.title('Bragg Curves — Proton in Water')
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Multi-Energy Comparison

Compare stopping power at multiple energies. Higher energy → lower
dE/dx at entrance (1/β² term) but deeper penetration.

In [ ]:
energies = np.logspace(0.5, 2.5, 20)  # 3–316 MeV
dedx = []
for E in energies:
    s = engine.compute_step(E, 0.01, 'water_icru', 'proton')
    dedx.append(s.dEdx_MeV_cm)

plt.figure(figsize=(8, 4))
plt.loglog(energies, dedx, '.-')
plt.xlabel('Kinetic Energy [MeV]')
plt.ylabel('dE/dx [MeV/cm]')
plt.title('Stopping Power: Proton in Water (Bethe-Bloch + Sternheimer)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. NIST PSTAR Validation

Compare our Bethe-Bloch implementation against NIST PSTAR reference
data. The ±2% target is met at energies ≥ 50 MeV.

In [ ]:
report = engine.validate_against_nist('water_icru', 'proton')
print(f"Validation: {'PASS' if report.passed else 'FAIL'}")
print(f"Reference:  {report.referenceSource}")
print(f"Deviations: {report.deviations}")

## 5. Profile-based Configuration

Use `ProfileManager` to load pre-configured analysis profiles.
Profiles are merged: default → profile → user overrides → CLI overrides.

In [ ]:
pm = beamlab.ProfileManager()
profiles = pm.available_profiles()
print(f"Available profiles: {profiles}")

# Resolve the 'quick_inspect' profile with a CLI override.
cfg = pm.resolve('quick_inspect')
print(f"Resolved config keys: {list(cfg.keys())[:6]}...")

## 6. Dataset Analysis Pipeline

Run the full analysis pipeline (import → analyse → export) on a CSV file.

**Note:** Replace `/data/muon_run_042.csv` with an actual Geant4 CSV file
or use the helpers to generate a synthetic dataset.

In [ ]:
# Generate a synthetic CSV for demonstration (10k samples).
def generate_synthetic_csv(path, n_traj=100, n_samples=100):
    import numpy as np
    np.random.seed(42)
    header = ('x_cm,y_cm,z_cm,edep_MeV,kinE_MeV,'
              'momx_MeV,momy_MeV,momz_MeV,time_ns,'
              'trackID,parentID,eventID,pdg,particleName,source_file')
    rows = []
    for t in range(n_traj):
        for s in range(n_samples):
            z = s / (n_samples - 1) * 30.0
            row = (f"{np.random.uniform(-5, 5):.4f},"
                   f"{np.random.uniform(-5, 5):.4f},"
                   f"{z * 100:.2f},0.010,100.0,10.0,0.0,0.0,"
                   f"{s * 0.1:.1f},{t + 1},0,1,13,mu-,synthetic")
            rows.append(row)
    with open(path, 'w') as f:
        f.write(header + '\n')
        f.write('\n'.join(rows))

import tempfile
csv_path = os.path.join(tempfile.mkdtemp(), 'synthetic_beam.csv')
generate_synthetic_csv(csv_path, 50, 100)
print(f"Generated {csv_path}")
print(f"  → {os.path.getsize(csv_path):,} bytes")

In [ ]:
# ── Run the pipeline ─────────────────────────────────────────────
# Requires AnalysisOrchestrator and ImporterRegistry.
#
# from beamlab_core import AnalysisOrchestrator, ImporterRegistry
#
# importers = ImporterRegistry()
# importers.register_importer(Geant4CsvImporter())
#
# orch = AnalysisOrchestrator(importers, ..., ...)
# result = orch.run(csv_path, cfg)
#
# print(f"Focus position: {result.focus_result.z_min:.3f} m")
# print(f"Envelope vertices: {len(result.envelope_geometry.vertices)}")

print("Pipeline execution requires the full AnalysisOrchestrator.")
print("See src/scripting/examples/advanced_pipeline.py for details.")

## 7. Custom Material

Add a custom material with user-defined properties and compute its
stopping power.

In [ ]:
my_mat = beamlab.Material()
my_mat.id = 'my_tissue'
my_mat.name = 'My Custom Tissue'
my_mat.density_g_cm3 = 1.05
my_mat.Z_eff = 7.5
my_mat.A_eff = 14.0
my_mat.meanExcitationEnergy_eV = 72.0

mats.add_custom(my_mat)
print(f"Custom material '{my_mat.id}' added.")

# Compute stopping power with the custom material.
step = engine.compute_step(150.0, 0.01, 'my_tissue', 'proton')
print(f"dE/dx in custom tissue: {step.dEdx_MeV_cm:.3f} MeV/cm")
print(f"  vs water: {engine.compute_step(150.0, 0.01, 'water_icru', 'proton').dEdx_MeV_cm:.3f} MeV/cm")

## 8. Performance Benchmark

Measure the time for a batch of simulations.

In [ ]:
import time

n_repeat = 50
start = time.perf_counter()
for _ in range(n_repeat):
    engine.compute_bragg_curve(150.0, 'water_icru', 'proton')
elapsed = time.perf_counter() - start

print(f"{n_repeat} Bragg curves: {elapsed:.2f}s total")
print(f"{elapsed / n_repeat * 1000:.1f} ms per curve")
print(f"Estimated speed: {elapsed / n_repeat / len(curve.depth_cm) * 1e6:.1f} μs/step")

---

*BeamLabStudio v3.0 — [GitHub](https://github.com/kegouro/BeamLabStudio)*